# Join Strategy Crossover and the Cost of a Stale Statistic

Companion notebook for the [Join Strategies: Broadcast, Shuffle Hash and Sort-Merge](https://codemia.io/courses/big_data_engines/bd_spark_join_strategies) lesson on Codemia.

**What you will build**: a cost model for the three join strategies, built on measured sort and hash throughput rather than
guesses. You will find the build side size where broadcast stops winning, discover that the memory
wall arrives first, see why engines default to sort-merge even though hashing is cheaper, and watch a
planner pick a plan that cannot run because a row count was out of date.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

EXECUTORS = 120
CORES_PER_EXEC = 4
CORES = EXECUTORS * CORES_PER_EXEC
EXEC_MEMORY_MB = 8_192                              # heap per executor
TASK_MEMORY_MB = 0.5 * EXEC_MEMORY_MB / CORES_PER_EXEC   # execution memory a task may use
NET_MB_S = 120                                      # per executor, a 1 Gbit link in practice
DISK_MB_S = 220                                     # local SSD, shuffle files
ROW_BYTES = 180                                     # average serialised row
HEAP_FACTOR = 2.5                                   # in-memory rows vs serialised bytes

print(f'{EXECUTORS} executors x {CORES_PER_EXEC} cores = {CORES} task slots')
print(f'{EXEC_MEMORY_MB / 1024:.0f} GB heap each, {TASK_MEMORY_MB:,.0f} MB per task, '
      f'{NET_MB_S} MB/s each, {DISK_MB_S} MB/s local disk')

## Concept recap

Three strategies, three different things to be afraid of.

A broadcast hash join sends the whole build side to every executor, builds a hash table there, and
streams the probe side past it with no shuffle at all. It is the fastest join there is when it fits,
and the cost is that the build side is copied once per executor.

A shuffle hash join partitions both sides by the join key, so each task gets one partition of each
side, builds a hash table from its share of the build side and probes it. It pays a full shuffle of
both sides and needs one partition of the build side to fit in a task's memory.

A sort-merge join partitions both sides the same way, sorts each partition, and walks the two sorted
runs together. It pays the shuffle and the sort, and in exchange it can spill to disk and keep going,
which is why engines default to it for large joins.

A planner chooses between them from row count and size estimates. The rest of this notebook measures
what the choices really cost and what happens when the estimate is wrong.

## Measuring the primitives

In [ ]:
def time_it(fn, repeats=3):
    fn()                                   # warm up, then take the best of several
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


sizes = [400_000, 800_000, 1_600_000, 3_200_000]
print(f'{"rows":>11}{"sort ns/row":>14}{"merge walk ns/row":>20}'
      f'{"hash build ns/row":>20}{"hash probe ns/row":>20}')
measured = []
for n in sizes:
    build = rng.integers(0, n, n)
    probe = rng.integers(0, n, n)
    sorted_build, sorted_probe = np.sort(build), np.sort(probe)
    t_sort = time_it(lambda: np.sort(build))
    t_walk = time_it(lambda: int((sorted_build == sorted_probe).sum()))
    keys, vals = build.tolist(), list(range(n))
    t_hash = time_it(lambda: dict(zip(keys, vals)), repeats=2)
    table = dict(zip(keys, vals))
    probe_list = probe.tolist()
    t_probe = time_it(lambda: [table.get(k) for k in probe_list], repeats=2)
    measured.append((n, t_sort, t_walk, t_hash, t_probe))
    print(f'{n:>11,}{t_sort / n * 1e9:>14.1f}{t_walk / n * 1e9:>20.1f}'
          f'{t_hash / n * 1e9:>20.1f}{t_probe / n * 1e9:>20.1f}')

ns = np.array([m[0] for m in measured], float)
print()
for label, idx in [('sort', 1), ('merge walk', 2), ('hash build', 3), ('hash probe', 4)]:
    ts = np.array([m[idx] for m in measured])
    slope = np.polyfit(np.log(ns), np.log(ts), 1)[0]
    print(f'{label:>11}: total time grows as n^{slope:.2f}')

Textbook complexity says hashing is linear and sorting is n log n, so a hash join should win every
time. The measurement says something more interesting. The sort is nearly linear in practice at these
sizes and costs tens of nanoseconds per row, the merge walk over two sorted arrays is a fraction of a
nanosecond per row because it is a sequential pass the memory system loves, and the hash operations
are both the most expensive per row and the ones whose per row cost grows as the input gets bigger.

The reason is locality rather than asymptotics. Every probe into a large hash table is a random
memory access, so as the table outgrows the caches each lookup costs more, while a sort does long
sequential runs throughout. That is why real engines invest so heavily in sorting and why the n log n
term matters less than the complexity classes suggest.

Two caveats on the numbers. These are one CPython thread, so the hash figures are an order of
magnitude rather than Spark's, and the division below makes that allowance explicit. The merge walk
is a compare pass that emits no rows, so treat it as the floor for the merge step rather than its
full cost.

In [ ]:
_, t_sort, t_walk, t_hash, t_probe = measured[-1]
n_big = measured[-1][0]
SORT_S_PER_ROW = t_sort / n_big
WALK_S_PER_ROW = t_walk / n_big
HASH_S_PER_ROW = t_hash / n_big / 4          # a JVM hash map, charitably 4x CPython
PROBE_S_PER_ROW = t_probe / n_big / 4

print(f'sort        {SORT_S_PER_ROW * 1e9:>7.1f} ns/row   (at {n_big:,} rows in a partition)')
print(f'merge walk  {WALK_S_PER_ROW * 1e9:>7.1f} ns/row')
print(f'hash build  {HASH_S_PER_ROW * 1e9:>7.1f} ns/row   (CPython measurement / 4)')
print(f'hash probe  {PROBE_S_PER_ROW * 1e9:>7.1f} ns/row')

## A cost model for the three strategies

In [ ]:
def mb(rows):
    return rows * ROW_BYTES / 1e6


def broadcast_cost(build_rows, probe_rows):
    """Collect the build side, fan it out torrent style, build a table per executor."""
    size_mb = mb(build_rows)
    rounds = int(np.ceil(np.log2(EXECUTORS)))          # peer to peer, not a star
    network = rounds * size_mb / NET_MB_S
    build = build_rows * HASH_S_PER_ROW                # on every executor, in parallel
    probe = probe_rows / CORES * PROBE_S_PER_ROW
    return {'strategy': 'broadcast', 'wall': network + build + probe,
            'shuffle_mb': 0.0, 'network_mb': size_mb * EXECUTORS,
            'exec_mem_mb': size_mb * HEAP_FACTOR,      # one shared copy per executor
            'task_mem_mb': 0.0, 'spilled_mb': 0.0, 'fails': False}


def one_shuffle(rows):
    """Serialise, write locally, fetch over the network, read back."""
    size_mb = mb(rows)
    disk = 2 * size_mb / (DISK_MB_S * EXECUTORS)
    net = size_mb * (1 - 1 / EXECUTORS) / (NET_MB_S * EXECUTORS)
    return disk + net, size_mb


def shuffle_hash_cost(build_rows, probe_rows, partitions=CORES):
    s1, mb1 = one_shuffle(build_rows)
    s2, mb2 = one_shuffle(probe_rows)
    waves = max(1.0, partitions / CORES)
    per_part_build = build_rows / partitions
    build = per_part_build * HASH_S_PER_ROW * waves
    probe = probe_rows / partitions * PROBE_S_PER_ROW * waves
    task_mem = mb(per_part_build) * HEAP_FACTOR       # one hash table per task
    return {'strategy': 'shuffle hash', 'wall': s1 + s2 + build + probe,
            'shuffle_mb': mb1 + mb2, 'network_mb': mb1 + mb2,
            'exec_mem_mb': task_mem * CORES_PER_EXEC,
            'task_mem_mb': task_mem, 'spilled_mb': 0.0,
            'fails': task_mem > TASK_MEMORY_MB}


def sort_merge_cost(build_rows, probe_rows, partitions=CORES):
    s1, mb1 = one_shuffle(build_rows)
    s2, mb2 = one_shuffle(probe_rows)
    waves = max(1.0, partitions / CORES)
    rows_per_part = (build_rows + probe_rows) / partitions
    sort = rows_per_part * SORT_S_PER_ROW * waves
    walk = rows_per_part * WALK_S_PER_ROW * waves
    task_mem = mb(rows_per_part) * HEAP_FACTOR
    spill_mb = 0.0
    extra = 0.0
    if task_mem > TASK_MEMORY_MB:                       # external sort: write runs, merge them
        spill_mb = mb(rows_per_part) * partitions
        extra = 2 * spill_mb / (DISK_MB_S * EXECUTORS)
    return {'strategy': 'sort-merge', 'wall': s1 + s2 + sort + walk + extra,
            'shuffle_mb': mb1 + mb2, 'network_mb': mb1 + mb2,
            'exec_mem_mb': min(task_mem, TASK_MEMORY_MB) * CORES_PER_EXEC,
            'task_mem_mb': task_mem, 'spilled_mb': spill_mb, 'fails': False}


BROADCAST_HEAP_LIMIT = 0.35 * EXEC_MEMORY_MB      # beyond this a broadcast is GC suicide


def feasible(cost):
    """Can this plan actually run? Sort-merge always can, the others have limits."""
    if cost['strategy'] == 'broadcast':
        return cost['exec_mem_mb'] <= BROADCAST_HEAP_LIMIT
    if cost['strategy'] == 'shuffle hash':
        return cost['task_mem_mb'] <= TASK_MEMORY_MB
    return True


PROBE_ROWS = 4_000_000_000        # a 720 GB fact table
print(f'probe side: {PROBE_ROWS:,} rows, {mb(PROBE_ROWS) / 1e3:.0f} GB, '
      f'{CORES} partitions')
print()
print(f'{"build rows":>14}{"build size":>12}{"broadcast":>12}{"bcast mem/exec":>16}'
      f'{"shuffle hash":>14}{"sort-merge":>12}{"best that fits":>16}')
for build_rows in [10_000, 100_000, 1_000_000, 5_000_000, 20_000_000,
                   100_000_000, 500_000_000, 4_000_000_000]:
    b = broadcast_cost(build_rows, PROBE_ROWS)
    h = shuffle_hash_cost(build_rows, PROBE_ROWS)
    m = sort_merge_cost(build_rows, PROBE_ROWS)
    ok = [c for c in (b, h, m) if feasible(c)]
    best = min(ok, key=lambda c: c['wall'])
    size = mb(build_rows)
    size_s = f'{size:.1f} MB' if size < 1000 else f'{size / 1e3:.1f} GB'
    bmem = b['exec_mem_mb']
    print(f'{build_rows:>14,}{size_s:>12}{b["wall"]:>10.1f} s'
          f'{bmem:>14,.0f} MB{h["wall"]:>12.1f} s{m["wall"]:>10.1f} s'
          f'{best["strategy"]:>16}')

Two things in that table are worth more than the rest. The first is how late the time crossover
arrives: broadcast is still the cheapest plan at a build side of several hundred megabytes, which is
tens of times past Spark's default ten megabyte broadcast threshold. The second is the memory column
next to it, which is why the default is set where it is.

Broadcast's wall clock grows linearly with the build side because a torrent broadcast still moves
every byte to every executor, while the shuffle strategies are dominated by the fact table and barely
notice the build side at all. So the two curves do cross, eventually, and long after the point where
the build side stops fitting in memory.

## The memory wall arrives before the time crossover

In [ ]:
print(f'{"build size":>12}{"copies":>8}{"cluster RAM":>14}{"per executor":>14}'
      f'{"% of heap":>11}{"verdict":>22}')
for build_rows in [50_000, 1_000_000, 10_000_000, 60_000_000, 200_000_000]:
    size = mb(build_rows)
    on_heap = size * HEAP_FACTOR
    frac = on_heap / EXEC_MEMORY_MB
    verdict = ('fine' if frac < 0.1 else
               'tight, watch GC' if frac < 0.35 else
               'spills or dies' if frac < 1 else 'OOM on every executor')
    print(f'{size:>10,.0f} MB{EXECUTORS:>8}{size * EXECUTORS / 1e3:>12,.1f} GB'
          f'{on_heap:>12,.0f} MB{frac:>11.1%}{verdict:>22}')

print()
print('the driver holds the whole thing before it ships a single byte:')
for build_rows in [1_000_000, 60_000_000]:
    print(f'  {mb(build_rows):>8,.0f} MB collected, '
          f'{mb(build_rows) * HEAP_FACTOR:>8,.0f} MB on the driver heap')
print()
safe_rows = 0.1 * EXEC_MEMORY_MB / HEAP_FACTOR * 1e6 / ROW_BYTES
print(f'a tenth of an executor heap is about {safe_rows / 1e6:.1f}M rows, '
      f'{0.1 * EXEC_MEMORY_MB / HEAP_FACTOR:,.0f} MB serialised,')
print(f'which is {0.1 * EXEC_MEMORY_MB / HEAP_FACTOR / 10:.0f}x the default 10 MB threshold.')

A broadcast number that people tune as a performance setting is really a memory decision. A sixty
million row dimension is eleven gigabytes of serialised rows, a terabyte and a half of duplicated
memory across the cluster, and several times an executor's heap on each one. The job does not get
slower, it fails, and it fails after the driver has already spent minutes collecting the table.

That asymmetry is the argument for being conservative. Below the line you give up some speed. Above
it you lose the job, and you lose it with an out of memory error that looks like a bug in your code
rather than like a planning decision.

In [ ]:
build_axis = np.logspace(4, 9.6, 40)
walls = {k: [] for k in ['broadcast', 'shuffle hash', 'sort-merge']}
for b in build_axis:
    for c in (broadcast_cost(b, PROBE_ROWS), shuffle_hash_cost(b, PROBE_ROWS),
              sort_merge_cost(b, PROBE_ROWS)):
        walls[c['strategy']].append(c['wall'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
for k, colour in [('broadcast', 'tab:green'), ('shuffle hash', 'tab:blue'),
                  ('sort-merge', 'tab:orange')]:
    ax1.plot(mb(build_axis), walls[k], color=colour, label=k)
ax1.axvline(10, color='black', linestyle=':', label='default 10 MB threshold')
ax1.axvline(0.1 * EXEC_MEMORY_MB / HEAP_FACTOR, color='tab:red', linestyle='--',
            label='10% of an executor heap')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('build side (MB, serialised)')
ax1.set_ylabel('modelled wall clock (s)')
ax1.set_title('Broadcast wins well past the default threshold')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3, which='both')

probe_axis = np.logspace(6, 9.8, 30)
for build_rows, colour in [(1_000_000, 'tab:green'), (50_000_000, 'tab:purple')]:
    ratio = [broadcast_cost(build_rows, p)['wall'] / sort_merge_cost(build_rows, p)['wall']
             for p in probe_axis]
    ax2.plot(probe_axis, ratio, color=colour, label=f'build = {build_rows:,} rows')
ax2.axhline(1, color='black', linestyle='--')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('probe side rows')
ax2.set_ylabel('broadcast wall / sort-merge wall')
ax2.set_title('The bigger the probe side, the more broadcast wins')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, which='both')
plt.show()

The right panel is the intuition to keep. Broadcast's advantage comes from not shuffling the probe
side, so it grows with the fact table: both curves sit below one and fall as the probe side grows,
and the fifty million row build side loses on memory rather than on time.

So the rule you can apply without a cost model is the memory one. Broadcast when the build side fits
comfortably inside a fraction of an executor heap, and shuffle when it does not. The red line on the
left panel is that rule, and it sits two orders of magnitude to the right of the default.

## Why the default is sort-merge and not shuffle hash

In [ ]:
BUILD_ROWS = 2_000_000_000       # both sides large: no broadcast in sight
print(f'both sides large: build {BUILD_ROWS:,} rows ({mb(BUILD_ROWS) / 1e3:.0f} GB), '
      f'probe {PROBE_ROWS:,} rows ({mb(PROBE_ROWS) / 1e3:.0f} GB)')
print(f'a task may use {TASK_MEMORY_MB:,.0f} MB of execution memory')
print()
print(f'{"partitions":>12}{"rows/task":>14}{"hash mem":>12}{"hash":>22}'
      f'{"sort mem":>11}{"sort-merge":>14}{"spilled":>12}')
for partitions in [240, 480, 2_000, 10_000, 40_000]:
    h = shuffle_hash_cost(BUILD_ROWS, PROBE_ROWS, partitions)
    m = sort_merge_cost(BUILD_ROWS, PROBE_ROWS, partitions)
    h_state = 'OOM' if h['fails'] else f'{h["wall"]:.0f} s'
    spill = f'{m["spilled_mb"] / 1e6:.1f} TB' if m['spilled_mb'] else '-'
    print(f'{partitions:>12,}{BUILD_ROWS / partitions:>14,.0f}'
          f'{h["task_mem_mb"]:>10,.0f} MB{h_state:>22}'
          f'{m["task_mem_mb"]:>9,.0f} MB{m["wall"]:>12.0f} s{spill:>12}')

print()
print('the same join, told as a question about failure modes:')
h = shuffle_hash_cost(BUILD_ROWS, PROBE_ROWS, 2_000)
m = sort_merge_cost(BUILD_ROWS, PROBE_ROWS, 2_000)
print(f'  shuffle hash at 2,000 partitions needs {h["task_mem_mb"]:,.0f} MB per task '
      f'and has {TASK_MEMORY_MB:,.0f} MB')
print(f'  sort-merge needs {m["task_mem_mb"]:,.0f} MB, spills what does not fit, '
      f'and finishes in {m["wall"]:.0f} s')

On wall clock alone the two shuffle strategies are close, because both are dominated by moving a
terabyte of rows across the network and onto disk, and the CPU terms we measured are seconds against
minutes. If the choice were about speed it would hardly matter which one an engine picked.

The choice is about what happens when a partition is bigger than a task's memory. The hash join has
no answer: the hash table either fits or the executor dies, and the only lever is more partitions,
which you have to get right before the job starts. The sort can always write sorted runs to disk and
merge them, so it degrades into a slower join instead of a failed one. That is why sort-merge is the
default for large joins and shuffle hash is an optimisation an engine applies only when it is
confident about the size.

## A planner, and a statistic that is out of date

In [ ]:
# the team raised this from the 10 MB default after a tuning post promised faster joins
BROADCAST_THRESHOLD_MB = 512.0


def plan(est_build_rows, probe_rows, threshold_mb=BROADCAST_THRESHOLD_MB):
    """A simplified planner: broadcast if the estimate fits, else the cheaper shuffle."""
    if mb(est_build_rows) <= threshold_mb:
        return 'broadcast'
    c = [shuffle_hash_cost(est_build_rows, probe_rows),
         sort_merge_cost(est_build_rows, probe_rows)]
    c = [x for x in c if not x.get('fails')] or [c[-1]]
    return min(c, key=lambda x: x['wall'])['strategy']


COSTS = {'broadcast': broadcast_cost, 'shuffle hash': shuffle_hash_cost,
         'sort-merge': sort_merge_cost}


def execute(strategy, build_rows, probe_rows):
    """What really happens, given the real row count."""
    cost = COSTS[strategy](build_rows, probe_rows)
    if strategy == 'broadcast':
        per_exec = cost['exec_mem_mb']
        if per_exec > EXEC_MEMORY_MB:
            return cost, f'OOM: {per_exec / 1e3:,.0f} GB/executor'
        if per_exec > BROADCAST_HEAP_LIMIT:
            return cost, f'heavy GC: {per_exec / 1e3:.1f} GB/executor'
    if cost.get('fails'):
        return cost, f'OOM: {cost["task_mem_mb"]:,.0f} MB/task'
    if cost['spilled_mb']:
        return cost, f'spills {cost["spilled_mb"] / 1e6:.1f} TB'
    return cost, 'ok'


def best_possible(build_rows, probe_rows):
    ok = [c for c in (broadcast_cost(build_rows, probe_rows),
                      shuffle_hash_cost(build_rows, probe_rows),
                      sort_merge_cost(build_rows, probe_rows)) if feasible(c)]
    return min(ok, key=lambda c: c['wall'])


SCENARIOS = [
    ('statistics are fresh',                    2_000_000,     2_000_000),
    ('table grew 150x since the last ANALYZE',  2_000_000,   300_000_000),
    ('filter guessed 10% selective, is 90%',    2_500_000,    22_500_000),
    ('no statistics, planner assumes small',       50_000,   900_000_000),
    ('estimate far too high, table is tiny',   40_000_000,        30_000),
]

print(f'broadcast threshold: {BROADCAST_THRESHOLD_MB:,.0f} MB '
      f'({BROADCAST_THRESHOLD_MB * 1e6 / ROW_BYTES / 1e6:.1f}M rows)')
print()
print(f'{"scenario":>42}{"estimate":>12}{"truth":>14}{"plan":>14}'
      f'{"outcome":>28}{"wall":>8}{"best":>8}')
for name, est, truth in SCENARIOS:
    strategy = plan(est, PROBE_ROWS)
    cost, outcome = execute(strategy, truth, PROBE_ROWS)
    ideal = best_possible(truth, PROBE_ROWS)
    wall = '-' if 'OOM' in outcome else f'{cost["wall"]:.0f} s'
    print(f'{name:>42}{est:>12,}{truth:>14,}{strategy:>14}'
          f'{outcome:>28}{wall:>8}{ideal["wall"]:>6.0f} s')

print()
print('the same five jobs with the 10 MB default:')
for name, est, truth in SCENARIOS:
    strategy = plan(est, PROBE_ROWS, threshold_mb=10.0)
    cost, outcome = execute(strategy, truth, PROBE_ROWS)
    wall = '-' if 'OOM' in outcome else f'{cost["wall"]:.0f} s'
    print(f'  {name:>42}  {strategy:>13}{outcome:>24}{wall:>8}')

Three of those five rows are the same bug with three different causes. The planner believed a small
number, chose broadcast, and the real table did not fit: once because nobody re ran ANALYZE after a
backfill, once because the estimate came out of a filter whose selectivity the planner guessed, and
once because there were no statistics at all and the default guess was optimistic. At planning time
the engine cannot tell these apart, which is why they produce the same executor out of memory error
and the same confused hour of debugging.

The block underneath is the uncomfortable half. The ten megabyte default rescues two of the three
crashes, because it simply refuses to broadcast a table whose estimate is a few hundred megabytes,
and the price is that the one job with correct statistics now runs about five times slower than it
needs to. Raising the threshold genuinely makes good plans faster. It also turns every bad estimate
from a slow job into a dead one, and you do not get to pick which of your estimates are bad.

The job with no statistics at all fails at either setting, because its guess is below every
threshold. That is the case no threshold can defend against, and the only fix is to make the estimate
exist: run ANALYZE, or let the engine see the real sizes at runtime, which is the next section.

The last row is the opposite mistake and it is the quiet one. An overestimate produces a correct but
needlessly expensive plan: a full shuffle of a fact table to join against thirty thousand lookup
rows, which runs fine and takes two orders of magnitude longer than it should. Nobody opens an
incident for a job that merely wastes money, which is exactly why these persist for months.

## Why runtime replanning fixes most of it

In [ ]:
print('adaptive execution plans the join after the first shuffle, '
      'when the real sizes are known:')
print()
print(f'{"truth":>14}{"static plan":>14}{"static outcome":>26}'
      f'{"adaptive plan":>15}{"adaptive wall":>15}')
for name, est, truth in SCENARIOS:
    static = plan(est, PROBE_ROWS)
    _, outcome = execute(static, truth, PROBE_ROWS)
    adaptive = plan(truth, PROBE_ROWS)             # planning on measured, not estimated, sizes
    a_cost, _ = execute(adaptive, truth, PROBE_ROWS)
    print(f'{truth:>14,}{static:>14}{outcome:>26}{adaptive:>15}{a_cost["wall"]:>13.0f} s')

print()
shuffle_s, shuffle_mb = one_shuffle(PROBE_ROWS)
print('the catch: the replan happens after both sides are already shuffled.')
print(f'for the 30,000 row case that is {shuffle_s:.0f} s and '
      f'{shuffle_mb / 1e3:.0f} GB of shuffle write')
print(f'that a correct estimate would have skipped entirely, against a '
      f'{best_possible(30_000, PROBE_ROWS)["wall"]:.1f} s broadcast.')

Adaptive planning removes the catastrophic failures, because by the time it chooses it is not
estimating anything: it has the actual bytes each shuffle produced. That is the single largest
reliability improvement in modern query engines, and it is why a stale statistic today is usually a
cost problem rather than an outage.

It is not a reason to stop maintaining statistics. The replan happens after the shuffle, so a join
that should have been a broadcast has already written the fact table to disk and read it back before
the engine notices anything. Good statistics stop the work from happening. Adaptive execution only
stops the crash.

## What to take away

- Hashing, probing and the merge walk are linear in rows; sorting is not. That single difference is
  the only real cost gap between the two shuffle joins, and it is small next to the shuffle itself.
- Broadcast wins because it does not shuffle the probe side, so its advantage grows with the fact
  table. Measured on time alone it stays the best plan far past the default ten megabyte threshold.
- That threshold is a memory limit wearing a performance costume. A broadcast duplicates the build
  side on every executor, inflated by object overhead, and the driver holds a copy first. Size it
  against a fraction of an executor heap, not against a benchmark.
- Engines default to sort-merge because it can spill. A shuffle hash join whose partition does not fit
  has no fallback, so the safe plan is the one that degrades instead of failing.
- A stale row count, a guessed selectivity and a missing statistic all produce the same crash: a
  broadcast that does not fit. An overestimate produces the quieter failure of a shuffle nobody
  needed, which can hide for months.
- Adaptive execution replans from real shuffle sizes and removes the crashes, but only after the
  shuffle is paid for.